# Detector-guided repair — Grounding DINO boxes prompt SAM 2.1 Hiera-L

Design note: `docs/repair_arm_design_note.md`. Pin: `docs/grounding_dino_pin.json`.

Model inference only. Frame selection, lifting, association, fusion, pooling
and evaluation all run locally in the repo.

**What changes from `repair_sam2_colab.ipynb`:** only the PROMPT. Same SAM 2.1
checkpoint, same 32 frames, same lifting. The automatic-mask arm seeded SAM
with a 32x32 point grid and got object parts; this one seeds it with detector
boxes for a fixed 41-class vocabulary.

**Pins.** SAM 2.1: sam2 @ `2b90b9f5ceec907a1c18123530e92e794ad901a4`,
checkpoint sha `2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318`.
Detector: `IDEA-Research/grounding-dino-base`. Thresholds `box_threshold=0.35`,
`text_threshold=0.25` — the published defaults, fixed before evaluation, **no
sweep**. Seeds 0.

**The detector revision and weight sha are not yet in the pin file.** Cell [1]
resolves and prints them; they are written into `docs/grounding_dino_pin.json`
by the freeze commit before any result is claimed. This is the same procedure
the SAM checkpoint went through in `docs/c1_p1_multiview_proposals_protocol.md`.

**Vocabulary is fixed and unmodified** — the 41 classes of
`extractors.learned_labels.GLOBAL_INDOOR_VOCABULARY_V1`, copied literally into
cell [2]. The local stage recomputes its hash and refuses a sidecar that used a
different list.

**Budget:** ONE run on 41069021. If it fails its gates locally, the repair
track closes — do not run 41069025, and do not touch 47331972.

In [ ]:
# [1] environment + both pinned checkpoints (verify BEFORE any inference)
import hashlib, glob, os
!git clone https://github.com/facebookresearch/sam2.git
%cd sam2
!git checkout 2b90b9f5ceec907a1c18123530e92e794ad901a4
!pip install -q -e .
!pip install -q --upgrade transformers
!wget -q -O sam2.1_hiera_large.pt https://dl.fbaipublicfiles.com/segment_anything_2/092824/sam2.1_hiera_large.pt
sam_sha = hashlib.sha256(open('sam2.1_hiera_large.pt','rb').read()).hexdigest()
assert sam_sha == '2647878d5dfa5098f2f8649825738a9345572bae2d4350a2468587ece47dd318', f'SAM CHECKPOINT SHA MISMATCH: {sam_sha}'
print('SAM checkpoint sha OK:', sam_sha)

# Detector: resolve the immutable revision, then download AT that revision.
from huggingface_hub import model_info, snapshot_download
GDINO_ID = 'IDEA-Research/grounding-dino-base'
GDINO_REVISION = model_info(GDINO_ID).sha            # record this
gdino_dir = snapshot_download(GDINO_ID, revision=GDINO_REVISION)
weights = sorted(glob.glob(os.path.join(gdino_dir, '*.safetensors'))) or \
          sorted(glob.glob(os.path.join(gdino_dir, '*.bin')))
assert len(weights) == 1, f'expected one weight file, found {weights}'
GDINO_SHA = hashlib.sha256(open(weights[0], 'rb').read()).hexdigest()
print('detector:', GDINO_ID)
print('  revision      :', GDINO_REVISION)
print('  weight file   :', os.path.basename(weights[0]))
print('  weight sha256 :', GDINO_SHA)
print('\n>>> PUT THESE TWO VALUES IN docs/grounding_dino_pin.json <<<')
import torch; print('torch', torch.__version__, '| cuda', torch.cuda.is_available())

In [ ]:
# [2] frames + the fixed 41-class vocabulary (copied verbatim; DO NOT EDIT)
SCENE_ID = 'arkitscenes_41069021'
BOX_THRESHOLD  = 0.35   # published default
TEXT_THRESHOLD = 0.25   # published default

VOCABULARY = [
    'armchair', 'bathtub', 'bed', 'bench', 'blinds', 'bookshelf', 'bottle',
    'bowl', 'box', 'cabinet', 'chair', 'clock', 'counter', 'cushion', 'desk',
    'door', 'drawer', 'indoor-plant', 'lamp', 'microwave', 'mirror', 'monitor',
    'nightstand', 'oven', 'picture', 'plate', 'plant-stand', 'projector',
    'refrigerator', 'rug', 'shelf', 'sink', 'sofa', 'stool', 'table', 'toilet',
    'trash-can', 'tv-monitor', 'vase', 'whiteboard', 'window',
]
assert len(VOCABULARY) == 41, len(VOCABULARY)
VOCAB_SHA = hashlib.sha256('\n'.join(VOCABULARY).encode()).hexdigest()
# Hyphens become spaces in the PROMPT only; the canonical form is stored.
PROMPT = ' . '.join(v.replace('-', ' ') for v in VOCABULARY) + ' .'
print('vocabulary sha:', VOCAB_SHA)

from google.colab import drive
drive.mount('/content/drive')
import tarfile, json
tar = f'/content/drive/MyDrive/repair/repair_frames_{SCENE_ID}.tar.gz'
tarfile.open(tar).extractall('/content/frames')
root = f'/content/frames/repair_frames_{SCENE_ID}'
manifest = json.load(open(os.path.join(root, 'upload_manifest.json')))
pngs = [os.path.join(root, f['png']) for f in manifest['frames']]
for f, p in zip(manifest['frames'], pngs):
    got = hashlib.sha256(open(p, 'rb').read()).hexdigest()
    assert got == f['sha256'], f"{f['png']}: sha256 mismatch"
SELECTION_SHA = manifest['selection_sha256']
assert SELECTION_SHA == '53982b5bf1166163f490cb2e4d6988e7b86e65d3e9033995d8dd51f572857363', SELECTION_SHA
print('frames ready:', len(pngs), '| selection', SELECTION_SHA[:16])

In [ ]:
# [3] detect, then segment each box. One pass, seeds 0, no retries.
import random, time, numpy as np, torch
from PIL import Image
random.seed(0); np.random.seed(0); torch.manual_seed(0)
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from sam2.build_sam import build_sam2
from sam2.sam2_image_predictor import SAM2ImagePredictor

processor = AutoProcessor.from_pretrained(GDINO_ID, revision=GDINO_REVISION)
detector = AutoModelForZeroShotObjectDetection.from_pretrained(
    GDINO_ID, revision=GDINO_REVISION).to('cuda').eval()
predictor = SAM2ImagePredictor(build_sam2(
    'configs/sam2.1/sam2.1_hiera_l.yaml', 'sam2.1_hiera_large.pt', device='cuda'))

out, t0 = {}, time.time()
torch.cuda.reset_peak_memory_stats()
n_boxes_total = 0
with torch.inference_mode(), torch.autocast('cuda', dtype=torch.bfloat16):
    for k, path in enumerate(pngs):
        image = Image.open(path).convert('RGB')
        w, h = image.size
        inputs = processor(images=image, text=PROMPT, return_tensors='pt').to('cuda')
        det = processor.post_process_grounded_object_detection(
            detector(**inputs), inputs.input_ids,
            threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
            target_sizes=[(h, w)])[0]
        boxes = det['boxes'].float().cpu().numpy().reshape(-1, 4)
        scores = det['scores'].float().cpu().numpy().reshape(-1)
        # transformers renamed this key across versions; accept either.
        phrases = det.get('text_labels', det.get('labels'))
        phrases = [str(p) for p in phrases]
        n_boxes_total += len(boxes)

        # One mask per box. multimask_output=False: the box already says which
        # object is meant, so there is nothing to disambiguate. NO background
        # or complement mask is ever produced.
        if len(boxes):
            predictor.set_image(np.array(image))
            masks, sam_scores, _ = predictor.predict(
                box=boxes, multimask_output=False)
            masks = np.asarray(masks).reshape(-1, h, w).astype(bool)
            sam_scores = np.asarray(sam_scores).reshape(-1)
            packed = np.stack([np.packbits(m.ravel()) for m in masks])
        else:
            packed = np.zeros((0, (h * w + 7) // 8), np.uint8)
            sam_scores = np.zeros((0,), np.float32)

        out[f'masks_{k:02d}']        = packed
        out[f'sam_scores_{k:02d}']   = np.asarray(sam_scores, dtype=np.float32)
        out[f'det_scores_{k:02d}']   = scores.astype(np.float32)
        out[f'boxes_{k:02d}']        = boxes.astype(np.float32)
        out[f'phrases_{k:02d}']      = np.array(phrases, dtype='<U64')
        out[f'shape_{k:02d}']        = np.array([h, w], dtype=np.int32)
        print(f'frame {k:02d}: {len(boxes)} boxes -> {len(packed)} masks  ({time.time()-t0:.0f}s)')
elapsed = time.time() - t0
peak = torch.cuda.max_memory_allocated() / 2**30
print(f'done in {elapsed:.0f}s, {n_boxes_total} boxes total, peak {peak:.2f} GiB')

In [ ]:
# [4] save sidecar to Drive
import platform
env = dict(scene_id=SCENE_ID, selection_sha256=SELECTION_SHA,
           arm='detector_guided',
           detector_model_id=GDINO_ID, detector_revision=GDINO_REVISION,
           detector_sha256=GDINO_SHA,
           box_threshold=BOX_THRESHOLD, text_threshold=TEXT_THRESHOLD,
           vocabulary=VOCABULARY, vocabulary_sha256=VOCAB_SHA,
           sam2_commit='2b90b9f5ceec907a1c18123530e92e794ad901a4',
           checkpoint_sha256=sam_sha, sam_prompt='box', multimask_output=False,
           background_or_complement_masks=False,
           torch=torch.__version__, cuda=torch.version.cuda,
           device=torch.cuda.get_device_name(0),
           python=platform.python_version(),
           elapsed_seconds=round(elapsed, 1), peak_vram_gib=round(peak, 2),
           n_frames=len(pngs), n_boxes_total=int(n_boxes_total), seeds=0)
dst = f'/content/drive/MyDrive/repair/repair_gdino_masks_{SCENE_ID}.npz'
np.savez_compressed(dst, env=json.dumps(env), **out)
print('saved:', dst)
print(json.dumps({k: v for k, v in env.items() if k != 'vocabulary'}, indent=1))

## After this notebook (local, in the repo)

1. Put `detector_revision` and `detector_sha256` from cell [1] into
   `docs/grounding_dino_pin.json` and commit — that is the freeze.
2. Download the sidecar to
   `runs/arkitscenes_repair/arkitscenes_41069021/repair_gdino_masks_arkitscenes_41069021.npz`.
3. Run:

```
python3 tools/arkitscenes_repair_propose_gdino.py --scene 41069021 \
    --masks runs/arkitscenes_repair/arkitscenes_41069021/repair_gdino_masks_arkitscenes_41069021.npz
python3 tools/arkitscenes_repair_eval.py --scene 41069021 \
    --repair runs/arkitscenes_repair/arkitscenes_41069021/repair_bank_gdino.npz
```

Zero unique IoU-0.50 recoveries closes the repair track.